In [2]:
!pip install -q transformers==4.40.0 datasets==2.19.0 peft==0.10.0 trl==0.8.6 accelerate==0.29.3 scipy
!pip install -q bitsandbytes --upgrade
!pip install -q triton

print("All dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 

In [3]:
from google.colab import drive
import torch

drive.mount('/content/drive')

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

Mounted at /content/drive
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
import json
from datasets import Dataset

DATASET_PATH    = '/content/drive/MyDrive/ctf_slm/prepared_data/train_formatted.jsonl'
MODEL_SAVE_PATH = '/content/drive/MyDrive/ctf_slm/model_checkpoints'

data = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                data.append(json.loads(line))
            except Exception:
                pass

dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} examples")
print("\nSample:")
print(dataset[0]['text'][:300])

Loaded 6681 examples

Sample:
### Instruction:
Give an example of a CTF methodology payload.

### Response:
# UofTCTF 2025 Write-up

University of Toronto CTF 2025

![image]([URL]

CTFtime link: [URL]

## WEB

### WEB - Scavenger Hunt

part - 1

Source code

![image]([URL]

part - 2

In response

![image]([URL]

part - 3

In Coo


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading {BASE_MODEL}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = prepare_model_for_kbit_training(model)

print(f'Model loaded!')
print(f'Params: {model.num_parameters()/1e9:.2f}B')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded!
Params: 1.10B
VRAM used: 1.04 GB


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338264987769031


In [8]:
import os
from transformers import TrainingArguments
from trl import SFTTrainer

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    evaluation_strategy='no',
    report_to='none',
    group_by_length=True,
    max_grad_norm=0.3,
    weight_decay=0.001,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field='text',
    tokenizer=tokenizer,
    max_seq_length=1024,
    packing=True,
)

print(f'Starting training on {len(dataset)} examples...')
print('Checkpoints saving to:', MODEL_SAVE_PATH)
trainer.train()
print('Training complete!')

Generating train split: 0 examples [00:00, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3993 > 2048). Running this sequence through the model will result in indexing errors
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting training on 6681 examples...
Checkpoints saving to: /content/drive/MyDrive/ctf_slm/model_checkpoints


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.103800
20,2.084000
30,1.976500
40,1.832800
50,1.940300
60,1.909200
70,1.798600
80,1.881200
90,1.788700
100,1.878600


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a n

Training complete!


In [9]:
from transformers import pipeline
import torch

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map='auto',
)

def ask(question, max_new_tokens=512):
    prompt = '### Instruction:\n' + question + '\n\n### Response:\n'
    output = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
    )
    return output[0]['generated_text'].split('### Response:\n')[-1].strip()

# Test with real CTF-style questions
tests = [
    "I found a login form. How do I test it for SQL injection?",
    "Binary has no canary and no PIE. What steps do I take to exploit it?",
    "I have a pcap file in a CTF forensics challenge. Walk me through analyzing it.",
    "What nmap command do I use to enumerate services on a target?",
    "I found an LFI vulnerability. How do I escalate it to RCE?",
]

for q in tests:
    print('\n' + '='*60)
    print('Q:', q)
    print('='*60)
    print(ask(q))

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo


Q: I found a login form. How do I test it for SQL injection?


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


[SQLi] username=username%3E&password=%3C
Username : username> 
Password : **********

```
#### So we can see that the server is sending us the username and password in html format so we need to make use of the **html-encode** function which will encode all HTML tags into their respective encoded values.

```ps1
python3 -m http.server 8000 --directory=/opt/lab_scripts
```

and then visiting the page

so i tried to execute this code using burp suite

![burp]([URL]

Q: Binary has no canary and no PIE. What steps do I take to exploit it?
- `printf` is the main function for printing strings. 
 - It takes two parameters: the string we want to print, and the output format (`%s`). The `%n` stands for "print newline". We pass a single argument, which will be printed on the screen (or in this case, an ASCII art of a cat).
 - A good example would be `printf("Hello world!");`. This prints the text Hello world! at the top of our terminal window.
 - For more information about printf and its syntax, 

In [10]:
import os

adapter_path = os.path.join(MODEL_SAVE_PATH, 'final_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Saved to: {adapter_path}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved to: /content/drive/MyDrive/ctf_slm/model_checkpoints/final_adapter
